# Naga quickstart

Run a bundled model, read the results, compare a scenario, run a small optimization, and build a model from Python. Every call here goes through the same headless engine as `naga run` and the Qt editor (`nagasim.api.run_model`).

## Run a bundled model

In [ ]:
from pathlib import Path

import nagasim
from nagasim import api

# A path relative to the installed package, not the working directory: works
# the same whether this notebook runs from the repo root or from Jupyter's
# own notebook folder.
model_path = str(Path(nagasim.__file__).parent / "models" / "single_server.json")
results = api.run_model(model_path, replications=5, run_length=200, warmup=20)
results

`results` is a `Results` -- a `dict[str, pandas.DataFrame]` (the same nine-plus KPI frames `run_model` always returned) with a few notebook conveniences on top. The table above is its `_repr_html_`.

In [ ]:
kpi_table = results.kpis()
kpi_table.head()

In [ ]:
ax = results.plot("system/Avg Queue Length")
ax

## Compare a scenario

`experiments.run_scenarios` runs every named scenario on the model's own resource CRN seeds and returns a paired-t comparison against the baseline.

In [ ]:
import json

from nagasim import experiments

layout = json.loads(open(model_path).read())
layout["simulation"]["time_limit"] = 200
layout["simulation"]["warmup"] = 20
layout["scenarios"] = [{
    "name": "Faster server",
    "overrides": [{"path": "blocks.process_1.params.dist_params.rate", "value": 1.6}],
}]

experiment = experiments.run_scenarios(layout, replications=5)
comparison = experiment.comparison(kpis=["system/Avg Queue Length", "system/Avg Throughput Time"])
comparison

## A small optimization

`optimize.optimize` searches one or more model parameters (`Factor`s) for the best value of a KPI, confirming the winner with extra replications.

In [ ]:
from nagasim.optimize import Factor, optimize

factors = [Factor(path="blocks.process_1.params.dist_params.rate", kind="continuous",
                  low=1.1, high=2.0, name="service_rate")]
opt = optimize(model_path, factors, response="system/Avg Queue Length", goal="min",
               budget=20, replications_per_point=2, run_length=200, warmup=20)
opt.best, opt.best_mean

## Build a model from Python

`ModelDocument` is the same object the Qt editor edits through; a script can build a model with it, save it to JSON, and run it exactly as if it had been drawn on the canvas.

In [ ]:
import tempfile
from pathlib import Path

from nagasim.core.document import ModelDocument

doc = ModelDocument()
doc.set_resources([{"name": "Server", "capacity": 1, "discipline": "FIFO"}])
doc.add_block("CREATE", (100, 100), "create_1")
doc.set_params("create_1", {"distribution": "Exponential", "dist_params": {"rate": 1.0}})
doc.add_block("PROCESS", (300, 100), "process_1")
doc.set_params("process_1", {"resource_name": "Server", "distribution": "Exponential",
                             "dist_params": {"rate": 1.25}})
doc.add_block("DISPOSE", (500, 100), "dispose_1")
doc.add_link("create_1", "process_1")
doc.add_link("process_1", "dispose_1")
doc.set_simulation(time_limit=200, warmup=20, replications=5)

tmp_model_path = Path(tempfile.gettempdir()) / "naga_quickstart_model.json"
doc.save(tmp_model_path)

doc_results = api.run_model(tmp_model_path)
doc_results.kpis().head()